<a href="https://colab.research.google.com/github/triastrale/analisis-kausal-penyebab-pembelian-impulsif/blob/main/src/dowhy/CausalInference_DoWhy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dowhy
!pip install networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 99.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: cvxpy
    Found existing installation: cvxpy 1.6.7
    Uninstalling cvxpy-1.6.7:
      Successfully uninstalled cvxpy-1.6.7


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from dowhy import gcm

In [ ]:
data = pd.read_excel('/content/drive/MyDrive/Histori Skripsi/DoWhy/BOSS3_graph_edges.xlsx')

data.head()

,index,Node 1,Interaction,Node 2,Ensemble,Edge,No Edge,--> dd pl,<-- dd pl,---,--> pd nl,<-- pd nl,--> dd nl,<-- dd nl,o->,<-o,o-o,<->
0,1,Gender,-->,H,0.987,0.987,0.013,NaN,NaN,NaN,NaN,NaN,0.987,NaN,NaN,NaN,NaN,NaN
1,2,SI,-->,SC,0.697,0.998,0.002,NaN,NaN,NaN,NaN,NaN,0.697,0.301,NaN,NaN,NaN,NaN
2,3,Educational Background,---,Job Status,0.669,1.000,NaN,0.242,NaN,0.669,NaN,0.089,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Educational Background,-->,E-Paylater User Status,0.658,0.658,0.342,0.658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Job Status,---,Monthly Income,0.618,1.000,NaN,NaN,0.129,0.618,NaN,NaN,0.253,NaN,NaN,NaN,NaN,NaN


DoWhy

In [ ]:
data_survei = pd.read_csv(
    '/content/drive/MyDrive/Histori Skripsi/DoWhy/Raw Data_Paylater and Non Paylater User_Average_Final.csv',
    sep=';',
    engine='python'
)

data_survei.head()

,Gender,E-Paylater User Status,Educational Background,Age,Job Status,Monthly Income,Average monthly expenditure for online shopping in relation to monthly income,IBB,P,SI,H,SC,NE
0,2,1,3,27,2,3,1,2.50,3.00,1.67,3.50,3.6,3.0
1,1,2,1,22,1,2,2,1.75,3.25,2.00,4.00,3.4,4.8
2,1,1,1,22,1,2,1,3.00,3.00,3.17,4.00,4.0,3.0
3,2,2,3,22,1,1,1,4.00,4.00,4.00,4.00,3.6,1.2
4,1,2,3,22,2,6,1,1.00,2.00,1.83,3.25,3.2,2.8


In [ ]:
causal_model = gcm.ProbabilisticCausalModel(
    nx.DiGraph([
        ('Gender', 'H'),
        ('Gender', 'Monthly Income'),
        ('SI', 'SC'),
        ('SI', 'IBB'),
        ('Educational Background', 'Job Status'),
        ('Educational Background', 'E-Paylater User Status'),
        ('Educational Background', 'SC'),
        ('Job Status', 'Monthly Income'),
        ('Job Status', 'E-Paylater User Status'),
        ('H', 'IBB'),
        ('H', 'SC'),
        ('P', 'SI'),
        ('P', 'E-Paylater User Status'),
        ('P', 'H'),
        ('IBB', 'SC'),
        ('IBB', 'NE'),
        ('Age', 'NE'),
        ('E-Paylater User Status', 'SI')
    ])
)

In [ ]:
gcm.auto.assign_causal_mechanisms(causal_model, data_survei)
gcm.fit(causal_model, data_survei)

probabilistic_causal_data = gcm.draw_samples(causal_model, num_samples=1000)
display(probabilistic_causal_data.head())

Fitting causal mechanism of node Age: 100%|██████████| 12/12 [00:00<00:00, 57.75it/s]


,Gender,Educational Background,P,Age,Job Status,H,Monthly Income,E-Paylater User Status,SI,IBB,SC,NE
0,2,1,3.0,25,0,3.965319,-2,1,2.923418,3.302842,3.679193,3.321325
1,1,4,1.5,22,2,3.857596,4,1,1.494262,2.538907,3.205878,3.762816
2,2,1,2.0,20,0,3.395919,1,2,2.384514,3.215272,3.632894,1.488102
3,2,1,1.0,26,0,3.732194,0,2,2.150152,2.286648,3.392805,3.415643
4,2,1,3.5,25,1,3.136165,2,1,2.151968,2.719003,3.349095,2.958497


In [ ]:
#Simulasi intervensi (Social Influence)
social_influence = gcm.interventional_samples(
    causal_model,
    {'SI': lambda x: x+1},
    num_samples_to_draw=1000
)

display(social_influence.head())

,Gender,Educational Background,P,Age,Job Status,H,Monthly Income,E-Paylater User Status,SI,IBB,SC,NE
0,2,3,1.00,46,2,3.289205,4,1,2.000000,1.690762,3.543763,4.003120
1,2,1,1.00,50,3,2.893287,4,1,3.273933,2.779580,3.709706,4.168446
2,2,3,3.75,36,2,3.711626,3,3,5.034478,3.755061,4.546893,2.628398
3,2,3,1.00,40,1,3.550325,1,1,3.594022,1.122451,3.033662,4.792972
4,1,1,1.00,23,1,2.877194,2,1,1.633630,2.240508,2.736755,2.502258


In [ ]:
comparison = pd.DataFrame({
    "Sebelum Intervensi": probabilistic_causal_data.mean(),
    "Sesudah Intervensi": social_influence.mean()
})

comparison

,Sebelum Intervensi,Sesudah Intervensi
Gender,1.659000,1.675000
Educational Background,2.226000,2.345000
P,2.344750,2.372500
Age,29.869000,29.780000
Job Status,1.610000,1.672000
H,3.485422,3.521912
Monthly Income,2.987000,3.086000
E-Paylater User Status,1.227000,1.271000
SI,2.375925,3.405829
IBB,2.779700,3.134364


In [ ]:
#Simulasi intervensi (Happiness)
happiness = gcm.interventional_samples(
    causal_model,
    {'H': lambda x: x+1},
    num_samples_to_draw=1000
)

display(happiness.head())

,Gender,Educational Background,P,Age,Job Status,H,Monthly Income,E-Paylater User Status,SI,IBB,SC,NE
0,1,3,3.75,21,2,4.618753,6,2,4.128487,3.565317,3.912711,2.591532
1,1,2,3.00,50,3,3.842823,5,2,2.439575,3.854079,3.844178,3.306070
2,2,1,4.25,29,1,4.487928,0,2,3.890634,3.525205,3.054039,3.661727
3,1,4,2.00,35,2,4.043338,5,1,1.341171,2.862383,3.799962,3.358156
4,2,1,1.00,21,1,4.620765,-1,1,3.273933,2.944599,4.028460,3.976781


In [ ]:
comparison = pd.DataFrame({
    "Sebelum Intervensi": probabilistic_causal_data.mean(),
    "Sesudah Intervensi": happiness.mean()
})

comparison

,Sebelum Intervensi,Sesudah Intervensi
Gender,1.659000,1.665000
Educational Background,2.226000,2.332000
P,2.344750,2.300000
Age,29.869000,29.889000
Job Status,1.610000,1.623000
H,3.485422,4.465893
Monthly Income,2.987000,2.966000
E-Paylater User Status,1.227000,1.272000
SI,2.375925,2.374946
IBB,2.779700,2.944790


In [ ]:
#Simulasi intervensi (H + SI)
happiness_social = gcm.interventional_samples(
    causal_model,
    {
        'H': lambda x: x+1,
        'SI': lambda x: x+1
    },
    num_samples_to_draw=1000
)

display(happiness_social.head())

,Gender,Educational Background,P,Age,Job Status,H,Monthly Income,E-Paylater User Status,SI,IBB,SC,NE
0,2,3,2.00,20,2,4.000000,4,1,2.000000,3.656392,3.167843,3.699534
1,1,3,2.00,23,3,5.029882,2,1,2.726067,2.684853,2.774782,4.430562
2,1,1,4.00,42,1,5.025937,3,1,5.137141,3.372641,4.408991,4.590232
3,2,1,1.75,43,1,5.081506,1,1,3.260935,3.874832,3.769129,2.146176
4,1,1,3.25,20,1,5.088349,2,2,3.170918,3.914913,3.763504,4.110752


In [ ]:
comparison = pd.DataFrame({
    "Sebelum Intervensi": probabilistic_causal_data.mean(),
    "Sesudah Intervensi": happiness_social.mean()
})

comparison

,Sebelum Intervensi,Sesudah Intervensi
Gender,1.659000,1.693000
Educational Background,2.226000,2.265000
P,2.344750,2.356250
Age,29.869000,29.960000
Job Status,1.610000,1.640000
H,3.485422,4.492255
Monthly Income,2.987000,2.916000
E-Paylater User Status,1.227000,1.254000
SI,2.375925,3.378138
IBB,2.779700,3.352000
